# Sesión 4 · Preprocesamiento de datos - SOLUCIÓN COMPLETA
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

**Objetivo:** Limpiar datos paso a paso con métodos detallados.

**Datos:** Dataset real de estación meteorológica desde GitHub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
pd.set_option('display.max_columns', 20)
print('✓ Librerías cargadas')

## 1. Cargar datos reales desde GitHub

In [ ]:
url = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_estacion.csv'
df = pd.read_csv(url)
print(f'✓ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nPrimeras filas:')
print(df.head())

## 2. Introducir problemas realistas

In [ ]:
df_limpio = df.copy()
df = df.copy()
rng = np.random.default_rng(42)

# Problemas
df.loc[rng.choice(len(df), 50, replace=False), 'irradiancia_wm2'] = np.nan
df.loc[rng.choice(len(df), 30, replace=False), 'temperatura_c'] = -999
df.loc[rng.choice(len(df), 15, replace=False), 'velocidad_viento_kmh'] *= 10
df = pd.concat([df, df.iloc[100:120]], ignore_index=True)

print('✓ Problemas introducidos')

## 6. ESTRATEGIAS DE IMPUTACIÓN - Análisis detallado

### ¿Qué es la imputación?
Es el proceso de **rellenar valores faltantes (NaN)** con estimaciones basadas en los datos disponibles.

### Métodos disponibles:

#### **1. ELIMINAR (dropna)**
- **Cómo funciona:** Simplemente elimina filas con NaN
- **Fórmula:** Si hay NaN, borra la fila
- **Cuándo usar:**
  - Datos con <5% faltantes
  - No crítico perder esas observaciones
  - Datos no temporal
- **Ventajas:** Simple, sin suposiciones
- **Desventajas:** Pierdes datos, rompe continuidad temporal
- **Para datos espaciales:** ❌ No recomendado (pierdes información)
- **Para series de tiempo:** ❌ Malo (rompe secuencia)

#### **2. MEDIA/MEDIANA**
- **Cómo funciona:** Rellena con el valor promedio (media) o central (mediana) de la columna
- **Fórmula:**
  - Media: `NaN = promedio de todos los valores`
  - Mediana: `NaN = valor del 50%`
- **Cuándo usar:**
  - Datos **independientes** (no temporal/espacial)
  - Distribución normal
- **Ventajas:** Rápido, mantiene media global
- **Desventajas:** Crea picos artificiales, pierde variación
- **Para datos espaciales:** ⚠️ Solo si datos muy cercanos
- **Para series de tiempo:** ❌ Malo (pierde variación horaria)

#### **3. INTERPOLACIÓN LINEAL**
- **Cómo funciona:** Dibuja una línea entre el punto anterior y siguiente, estima el valor en el medio
- **Fórmula:** 
  ```
  valor_faltante = valor_anterior + (valor_siguiente - valor_anterior) / 2
  ```
  Generalizado para cualquier posición t entre dos puntos.
- **Cuándo usar:**
  - **Series de tiempo** ✅ (datos meteorológicos, energía)
  - Variación suave entre valores
  - Datos espaciales donde vecinos son relevantes ✅
- **Ventajas:** Mantiene forma natural, respeta tendencia
- **Desventajas:** Asume cambio lineal (no siempre cierto)
- **Para datos espaciales:** ✅ Excelente (usa gradientes)
- **Para series de tiempo:** ✅ Recomendado

#### **4. INTERPOLACIÓN POLINOMIAL**
- **Cómo funciona:** Ajusta un polinomio (curva) a los datos, estima el valor faltante en esa curva
- **Fórmula:** y = a + bx + cx² + dx³ + ...
- **Cuándo usar:**
  - Datos con **curvas suaves** (no lineales)
  - Radiación solar (forma parabólica)
  - Temperatura diaria (forma sinusoidal)
- **Ventajas:** Ajusta curvas complejas mejor
- **Desventajas:** Puede crear oscilaciones, más datos necesarios
- **Para datos espaciales:** ✅ Bueno (si hay curvatura)
- **Para series de tiempo:** ✅ Muy bueno (captura patrones)

#### **5. FORWARD FILL (ffill) / BACKWARD FILL (bfill)**
- **Cómo funciona:** Copia el último valor conocido hacia adelante (ffill) o hacia atrás (bfill)
- **Fórmula:** 
  - ffill: `NaN = valor_anterior` (repite)
  - bfill: `NaN = valor_siguiente` (copia hacia atrás)
- **Cuándo usar:**
  - Datos que **no cambian mucho** entre observaciones
  - Categorías (no cambio esperado)
  - Últimos valores en series de tiempo
- **Ventajas:** Mantiene valores reales (no estima)
- **Desventajas:** Asume constancia, no interpola realmente
- **Para datos espaciales:** ⚠️ Solo para categorías
- **Para series de tiempo:** ⚠️ Solo para estaciones cortas

#### **6. SPLINE (Interpolación suave)**
- **Cómo funciona:** Ajusta curvas suaves (splines) que pasan por los puntos conocidos
- **Fórmula:** Polinomios cúbicos por tramos (cubicspline)
- **Cuándo usar:**
  - Datos muy suaves con pocas discontinuidades
  - Radiación solar, temperatura
  - Interpolación de imágenes
- **Ventajas:** Muy suave, sigue patrones naturales
- **Desventajas:** Más compleja computacionalmente
- **Para datos espaciales:** ✅ Excelente (2D/3D)
- **Para series de tiempo:** ✅ Muy bueno

In [ ]:
# COMPARAR MÉTODOS
col_ejemplo = 'irradiancia_wm2'
valores_con_huecos = df[col_ejemplo].copy()

# Aplicar 5 métodos
s1 = valores_con_huecos.dropna()
s2 = valores_con_huecos.fillna(valores_con_huecos.median())
s3 = valores_con_huecos.interpolate(method='linear')
s4 = valores_con_huecos.interpolate(method='polynomial', order=2)  # Polinomial orden 2
s5 = valores_con_huecos.fillna(method='ffill').fillna(method='bfill')

print('COMPARACIÓN DE 5 MÉTODOS DE IMPUTACIÓN:')
print(f'\n1. ELIMINAR: {len(s1)} datos (perdemos {len(valores_con_huecos)-len(s1)})')
print(f'2. MEDIANA: {len(s2)} datos')
print(f'3. LINEAL: {len(s3)} datos')
print(f'4. POLINOMIAL (orden 2): {len(s4)} datos')
print(f'5. FORWARD FILL: {len(s5)} datos')

In [ ]:
# Visualizar todos los métodos
rango = slice(0, 300)
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('COMPARACIÓN DE 6 MÉTODOS DE IMPUTACIÓN (Irradiancia Solar)', fontsize=14, fontweight='bold')

# 1. Datos con huecos
ax = axes[0, 0]
ax.plot(valores_con_huecos[rango].values, color='blue', alpha=0.6, linewidth=1)
ax.scatter(valores_con_huecos[rango][valores_con_huecos[rango].isna()].index, 
          [0]*valores_con_huecos[rango].isna().sum(), color='red', s=30, label='NaN')
ax.set_title('0. DATOS CON HUECOS (original)', fontweight='bold')
ax.set_ylabel('W/m²')
ax.grid(True, alpha=0.3)
ax.legend()

# 2. Eliminar
ax = axes[0, 1]
s1_rango = s1[rango]
ax.scatter(s1_rango.index, s1_rango.values, color='red', s=10, alpha=0.6)
ax.set_title('1. ELIMINAR (Pierdes 50 datos)', fontweight='bold', color='darkred')
ax.set_ylabel('W/m²')
ax.grid(True, alpha=0.3)

# 3. Mediana
ax = axes[1, 0]
s2_rango = s2[rango]
ax.plot(s2_rango.index, s2_rango.values, color='orange', marker='o', markersize=2, linewidth=1)
ax.axhline(valores_con_huecos.median(), color='orange', linestyle='--', alpha=0.5)
ax.set_title(f'2. MEDIANA ({valores_con_huecos.median():.0f}) - Crea picos planos', fontweight='bold', color='darkorange')
ax.set_ylabel('W/m²')
ax.grid(True, alpha=0.3)

# 4. Lineal
ax = axes[1, 1]
s3_rango = s3[rango]
ax.plot(s3_rango.index, s3_rango.values, color='green', marker='^', markersize=2, linewidth=1.5)
ax.scatter(valores_con_huecos[rango][valores_con_huecos[rango].isna()].index,
          s3_rango[valores_con_huecos[rango].isna()].values,
          color='darkgreen', s=50, marker='D', edgecolors='black', linewidth=1)
ax.set_title('3. INTERPOLACIÓN LINEAL ✓ (Mantiene forma)', fontweight='bold', color='darkgreen')
ax.set_ylabel('W/m²')
ax.grid(True, alpha=0.3)

# 5. Polinomial
ax = axes[2, 0]
s4_rango = s4[rango]
ax.plot(s4_rango.index, s4_rango.values, color='purple', marker='s', markersize=2, linewidth=1.5, label='Polinomial (orden 2)')
ax.scatter(valores_con_huecos[rango][valores_con_huecos[rango].isna()].index,
          s4_rango[valores_con_huecos[rango].isna()].values,
          color='purple', s=50, marker='D', edgecolors='black', linewidth=1, label='Rellenados')
ax.set_title('4. POLINOMIAL (Sigue curvas mejor)', fontweight='bold', color='purple')
ax.set_ylabel('W/m²')
ax.set_xlabel('Índice')
ax.grid(True, alpha=0.3)
ax.legend()

# 6. Forward Fill
ax = axes[2, 1]
s5_rango = s5[rango]
ax.plot(s5_rango.index, s5_rango.values, color='brown', marker='*', markersize=3, linewidth=1, linestyle='--')
ax.set_title('5. FORWARD FILL (Repite valores, por escalones)', fontweight='bold', color='brown')
ax.set_ylabel('W/m²')
ax.set_xlabel('Índice')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('\n✓ Visualización comparativa de todos los métodos')

## Recomendaciones por tipo de datos:

| Tipo de datos | Mejor método | Razón |
|---|---|---|
| **Series de tiempo** | Interpolación lineal/spline | Respeta tendencia temporal |
| **Datos espaciales 2D** | Spline 2D | Interpolación suave de superficies |
| **Datos independientes** | Media/Mediana | No hay relación entre observaciones |
| **Datos con curvas** | Polinomial | Captura no-linealidad |
| **Cambios lentos** | Forward fill | Valor previo relevante |
| **Radiación solar** | Polinomial (orden 2-3) | Forma parabólica natural |
| **Temperatura** | Spline/Polinomial | Variación suave |
| **Viento** | Lineal | Cambios menos predecibles |

## 9. ESCALADO - MÉTODOS DETALLADOS

### ¿Por qué escalar?
Variables con rangos muy distintos (irradiancia 0-900, viento 0-10) pueden confundir modelos ML.

### MÉTODO 1: StandardScaler (Estandarización)
- **Fórmula:** `z = (x - media) / desv_est`
- **Resultado:** Media=0, Desv.est=1
- **Cuándo usar:** Modelos que asumen distribución normal (regresión, NN)
- **Ventajas:** Mantiene relación entre valores
- **Desventajas:** No acota a rango específico

### MÉTODO 2: MinMaxScaler (Normalización)
- **Fórmula:** `x_scaled = (x - min) / (max - min)`
- **Resultado:** Todos los valores entre 0 y 1
- **Cuándo usar:** Redes neuronales, cuando necesitas acotación
- **Ventajas:** Acota a [0,1], intuitivo
- **Desventajas:** Sensible a outliers

### MÉTODO 3: RobustScaler
- **Fórmula:** `x_scaled = (x - Q2) / (Q3 - Q1)`  (usa mediana e IQR)
- **Resultado:** Robusto a outliers
- **Cuándo usar:** Datos con muchos outliers

### MÉTODO 4: LogScaler
- **Fórmula:** `x_scaled = log(x)`
- **Resultado:** Comprime valores grandes
- **Cuándo usar:** Datos con distribución log-normal (muchas ciencias)

In [ ]:
# Preparar datos para escalado
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'], format='mixed', errors='coerce')
df['temperatura_c'] = df['temperatura_c'].replace(-999, np.nan)
df = df.drop_duplicates().reset_index(drop=True)

col_test = 'irradiancia_wm2'
valores_originales = df[col_test].dropna()

# Aplicar 4 métodos de escalado
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# 1. Standard
scaler_std = StandardScaler().fit(valores_originales.values.reshape(-1, 1))
s1_std = scaler_std.transform(valores_originales.values.reshape(-1, 1)).flatten()

# 2. MinMax
scaler_mm = MinMaxScaler().fit(valores_originales.values.reshape(-1, 1))
s2_mm = scaler_mm.transform(valores_originales.values.reshape(-1, 1)).flatten()

# 3. Robust
scaler_rb = RobustScaler().fit(valores_originales.values.reshape(-1, 1))
s3_rb = scaler_rb.transform(valores_originales.values.reshape(-1, 1)).flatten()

# 4. Log
s4_log = np.log1p(valores_originales.values)  # log(1+x) para evitar log(0)

print('COMPARACIÓN DE 4 MÉTODOS DE ESCALADO:')
print(f'\nValores originales:')
print(f'  Min: {valores_originales.min():.2f}, Max: {valores_originales.max():.2f}')
print(f'  Media: {valores_originales.mean():.2f}, Desv.est: {valores_originales.std():.2f}')

print(f'\n1. StandardScaler (z-score):')
print(f'  Min: {s1_std.min():.2f}, Max: {s1_std.max():.2f}')
print(f'  Media: {s1_std.mean():.2f}, Desv.est: {s1_std.std():.2f}')

print(f'\n2. MinMaxScaler [0,1]:')
print(f'  Min: {s2_mm.min():.2f}, Max: {s2_mm.max():.2f}')
print(f'  Media: {s2_mm.mean():.2f}, Desv.est: {s2_mm.std():.2f}')

print(f'\n3. RobustScaler (IQR):')
print(f'  Min: {s3_rb.min():.2f}, Max: {s3_rb.max():.2f}')
print(f'  Media: {s3_rb.mean():.2f}, Desv.est: {s3_rb.std():.2f}')

print(f'\n4. LogScaler (log(1+x)):')
print(f'  Min: {s4_log.min():.2f}, Max: {s4_log.max():.2f}')
print(f'  Media: {s4_log.mean():.2f}, Desv.est: {s4_log.std():.2f}')

In [ ]:
# Visualizar todos los escaladores
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('COMPARACIÓN DE 4 MÉTODOS DE ESCALADO (Irradiancia)', fontsize=14, fontweight='bold')

# Original
ax = axes[0, 0]
ax.hist(valores_originales, bins=30, color='blue', alpha=0.7, edgecolor='black')
ax.set_title('0. ORIGINAL\nMin=0, Max=896, Media=213', fontweight='bold')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Standard
ax = axes[0, 1]
ax.hist(s1_std, bins=30, color='green', alpha=0.7, edgecolor='black')
ax.set_title('1. STANDARD (z-score)\nMedia≈0, Desv.est≈1\nPara: Regresión, NN', fontweight='bold', color='darkgreen')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# MinMax
ax = axes[0, 2]
ax.hist(s2_mm, bins=30, color='orange', alpha=0.7, edgecolor='black')
ax.set_title('2. MINMAX [0,1]\nAcota a [0,1]\nPara: Redes neuronales', fontweight='bold', color='darkorange')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Robust
ax = axes[1, 0]
ax.hist(s3_rb, bins=30, color='purple', alpha=0.7, edgecolor='black')
ax.set_title('3. ROBUST (IQR)\nResistente a outliers\nPara: Datos con extremos', fontweight='bold', color='purple')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Log
ax = axes[1, 1]
ax.hist(s4_log, bins=30, color='brown', alpha=0.7, edgecolor='black')
ax.set_title('4. LOG (log(1+x))\nComprime valores grandes\nPara: Dist. log-normal', fontweight='bold', color='brown')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Comparación visual
ax = axes[1, 2]
ax.boxplot([valores_originales/100, s1_std, s2_mm, s3_rb], 
          labels=['Original/100', 'Standard', 'MinMax', 'Robust'])
ax.set_title('Comparación de distribuciones', fontweight='bold')
ax.set_ylabel('Valor')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('\n✓ Visualización de todos los escaladores')

## TABLA FINAL: MÉTODOS SEGÚN CONTEXTO

### IMPUTACIÓN:
| Datos | Mejor método | Alternativa | Evitar |
|-------|--------------|------------|--------|
| Series de tiempo | Lineal / Spline | Polinomial | Mediana, Eliminar |
| Datos espaciales 2D | Spline 2D | Lineal | Mediana |
| Datos meteorológicos | Polinomial orden 2-3 | Lineal | Eliminar |
| Cambios lentos | Forward fill | Media | No relación |

### ESCALADO:
| Modelo | Mejor método | Razón |
|--------|--------------|-------|
| Regresión lineal | StandardScaler | Asume distribución normal |
| Redes neuronales | MinMaxScaler | Necesita rango [0,1] |
| Datos con outliers | RobustScaler | No sensible a extremos |
| Distribución log-normal | LogScaler | Normaliza distribución |

### DETECCIÓN DE ATÍPICOS:
| Método | Cuándo usar | Sensibilidad |
|--------|------------|---------------|
| IQR (Tukey) | Distribución normal | Moderada |
| Z-score | Datos normales | Alta |
| Isolation Forest | No-normal, complejos | Adaptativa |
| MAD (Median Abs Dev) | Robusta | Baja |